# Imports

In [17]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

from CustomLayers import DummyLinear
from benchmark.perplexity import measure_ppl

# Load data

In [ ]:
# raw_datasets = load_dataset("Salesforce/wikitext", "wikitext-103-raw-v1", split="test") # longer sequnces
raw_datasets = load_dataset("zhengxuanzenwu/wikitext-2-split-128", split="test")

Repo card metadata block was not found. Setting CardData to empty.
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Generating test split: 100%|██████████| 8192/8192 [00:00<00:00, 1637971.99 examples/s]


In [18]:
prompts = [x['text'] for x in raw_datasets if len(x['text']) > 0]

print('Number of sequences:', len(prompts))

Number of sequences: 8192


# Load model

In [7]:
model_id = "unsloth/Llama-3.2-1B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto")

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


# Get customized model

In [ ]:
custom_model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto")

def change_linear_layer(model, new_layer):
    for layer in model.model.layers:
        layer.self_attn.q_proj = new_layer(layer.self_attn.q_proj)
        layer.self_attn.k_proj = new_layer(layer.self_attn.k_proj)
        layer.self_attn.v_proj = new_layer(layer.self_attn.v_proj)
        layer.self_attn.o_proj = new_layer(layer.self_attn.o_proj)

        layer.mlp.gate_proj = new_layer(layer.mlp.gate_proj)
        layer.mlp.up_proj = new_layer(layer.mlp.up_proj)
        layer.mlp.down_proj = new_layer(layer.mlp.down_proj)

    model.lm_head = new_layer(model.lm_head)

    return model

custom_model = change_linear_layer(custom_model, DummyLinear)

# Measure Perplexity and Time

In [ ]:
orig_ppl, orig_time = measure_ppl(prompts, model, tokenizer)
custom_ppl, custom_time = measure_ppl(prompts, custom_model, tokenizer) # посчитать в колабе (там нормально бежит) !

100%|██████████| 8192/8192 [03:44<00:00, 36.57it/s]



Perplexity: 345.0567
Mean time per sample: 0.027 s


  1%|          | 71/8192 [02:07<4:03:44,  1.80s/it]


KeyboardInterrupt: 